# COMP3122 — Lab 04 — Series, DataFrames and Loading Data
### Week 4 · Chapter 4: Pandas — Series, DataFrames, and Loading Data

**Applied Data Science**

**This lab is graded.** It covers Chapter 4 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**One rule for the whole lab: no `for` loops and no list comprehensions.** Every task can be done with a Pandas expression. Finding that expression is the skill being assessed.

**The checker looks at your labels, not only your numbers.** Right values under the wrong row or column labels will not pass — that is the whole point of this chapter.

Work in order. The setup cell builds `weather` and `enrollments` and writes the four CSV files Part 5 reads, so run it first. If a check will not pass after five minutes, ask.

Part 3 is the one to slow down for. Before writing anything there, ask of every bracket: is the thing inside it a **label** or a **position**?

---
## Setup — run this first

In [1]:
# Run this cell once, before you start. It builds the data and defines the checker.
# The weather and enrollments tables are constructed for this lab (illustrative values, chosen so
# every check has one clean answer); Part 7 is the real dataset.
import pandas as pd
import numpy as np
from pathlib import Path

weather = pd.DataFrame(
    {
        "Mon": [22, 30, 18, 28],
        "Tue": [24, 31, 19, 27],
        "Wed": [25, 33, 19, 26],
        "Thu": [27, 34, 20, 25],
        "Fri": [26, 33, 21, 26],
        "Sat": [25, 32, 20, 27],
        "Sun": [23, 30, 19, 28],
    },
    index=["Toronto", "Cairo", "Tokyo", "Sydney"],
)

enrollments = pd.DataFrame({
    "course":   ["COMP3122", "COMP3122", "COMP1011", "COMP1011", "COMP2152", "COMP2152"],
    "section":  ["A", "B", "A", "B", "A", "B"],
    "enrolled": [38, 42, 35, 28, 40, 22],
    "capacity": [40, 40, 40, 40, 40, 40],
    "delivery": ["in-person", "in-person", "online", "online", "hybrid", "hybrid"],
})

cities = ["Toronto", "Cairo", "Tokyo", "Sydney"]
days   = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# The four CSV files Part 5 reads. Written here so the lab runs anywhere, with no
# separate download and no FileNotFoundError halfway through.
Path("weather.csv").write_text(
    "city,Mon,Tue,Wed,Thu,Fri,Sat,Sun\n"
    "Toronto,22,24,25,27,26,25,23\n"
    "Cairo,30,31,33,34,33,32,30\n"
    "Tokyo,18,19,19,20,21,20,19\n"
    "Sydney,28,27,26,25,26,27,28\n")
Path("euro_format.csv").write_text(
    "city;temperature;humidity\nParis;18;72\nBerlin;15;80\nMadrid;24;58\nRome;22;65\n")
Path("with_preamble.csv").write_text(
    "# Source: values constructed for COMP3122 Chapter 4 (illustrative, not observations)\n"
    "# Note: values are daily highs in degrees Celsius\n"
    "city,Mon,Tue,Wed\nToronto,22,24,25\nCairo,30,31,33\n"
    "Tokyo,18,19,19\nSydney,28,27,26\n")
Path("messy.csv").write_text(
    "city,Mon,Tue,Wed\nToronto,22,24,25\nCairo,30,unknown,33\n"
    "Tokyo,missing,19,19\nSydney,28,27,unknown\n")


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used where the check has to do something to your answer first (take
    its shape, round it, index into it). Before the answer is filled in that
    would raise, and a traceback where a hint should be is not helpful — so it
    is caught and reported as NOT YET instead.

    Pandas objects are compared on THREE things: the values, the row labels, and
    (for a DataFrame) the column labels. Getting the right numbers under the
    wrong labels is not a right answer in this chapter — that is the whole point
    of Chapter 4 — so the checker refuses to ignore them.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"        your answer raised {type(e).__name__}: {e}")
            return

    def _fail(why=None):
        print(f"NOT YET  {label}")
        if why:
            print(f"        {why}")
        print(f"        you gave: {got!r}")
        print(f"        expected: {want!r}")

    PANDAS = (pd.Series, pd.DataFrame)
    if isinstance(want, PANDAS) or isinstance(got, PANDAS):
        if type(got) is not type(want):
            return _fail(f"wrong type — got {type(got).__name__}, expected {type(want).__name__}")
        if list(got.index) != list(want.index):
            return _fail("the row labels do not match")
        if isinstance(want, pd.DataFrame) and list(got.columns) != list(want.columns):
            return _fail("the column labels do not match")
        try:
            ok = bool(got.equals(want)) or bool(np.allclose(
                got.to_numpy(dtype=float), want.to_numpy(dtype=float)))
        except Exception:
            ok = False
    else:
        g, w = np.asarray(got), np.asarray(want)
        try:
            if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
                ok = bool(np.allclose(g, w))   # decimals: allow a tiny rounding difference
            else:
                ok = bool(np.array_equal(g, w))
        except Exception:
            ok = False

    if ok:
        print(f"PASS  {label}")
    else:
        _fail()


print("Ready. weather:", weather.shape, " enrollments:", enrollments.shape,
      " pandas", pd.__version__)


Ready. weather: (4, 7)  enrollments: (6, 5)  pandas 3.0.6


---
## Part 1 — The Series  ·  §4.2

A Series is a labelled 1D array. Everything here is about the labels, not the numbers.

**Task 1.1.** Build a Series called `t11` holding Cairo's seven daily highs
`[30, 31, 33, 34, 33, 32, 30]`, indexed by the day names in `days`, and named `"Cairo"`.

In [2]:
t11 = pd.Series(data=np.array([30,31,33,34,33,32,30]), index=days, name="Cairo")

In [4]:
check('1.1 values', lambda: t11.tolist(), [30, 31, 33, 34, 33, 32, 30])
check('1.1 index',  lambda: list(t11.index), days)
check('1.1 name',   lambda: t11.name, 'Cairo')

PASS  1.1 values
PASS  1.1 index
PASS  1.1 name


**Task 1.2.** From `t11`, put Thursday's value in `t12_label` **using the label**, and the
same value in `t12_pos` **using the position**. Do not type `34`.

In [3]:
t12_label = t11.Thu
t12_pos   = t11[t11 == np.max(t11)].values[0]

In [5]:
check('1.2 by label',    t12_label, 34)
check('1.2 by position', t12_pos, 34)

PASS  1.2 by label
PASS  1.2 by position


**Task 1.3.** Slice `t11` two ways and store the **number of values** each returns.
`t13_label` from the label slice `"Tue"` to `"Fri"`. `t13_pos` from the position slice
`1` to `4`. Predict both before you run it.

In [7]:
t13_label = len(t11.loc["Tue" : "Fri"])
t13_pos   = len(t11[1:4])

In [8]:
check('1.3 label slice length',    t13_label, 4)
check('1.3 position slice length', t13_pos, 3)

PASS  1.3 label slice length
PASS  1.3 position slice length


**Task 1.4.** Put Cairo's weekly mean, rounded to 2 decimals, in `t14_mean`; the hottest
value in `t14_max`; and the whole Series converted to Fahrenheit (`c * 9 / 5 + 32`) in
`t14_f`. `t14_f` must still be labelled by day.

In [9]:
t14_mean = np.round(np.mean(weather.loc["Cairo"]),2)
t14_max  = np.max(weather.loc["Cairo"])
t14_f    = weather.loc["Cairo"] * (9 / 5) +32

In [10]:
check('1.4 mean', t14_mean, 31.86)
check('1.4 max',  t14_max, 34)
check('1.4 fahrenheit index', lambda: list(t14_f.index), days)
check('1.4 fahrenheit values', lambda: [round(v,1) for v in t14_f.tolist()],
      [86.0, 87.8, 91.4, 93.2, 91.4, 89.6, 86.0])

PASS  1.4 mean
PASS  1.4 max
PASS  1.4 fahrenheit index
PASS  1.4 fahrenheit values


---
## Part 2 — The DataFrame  ·  §4.3

The table itself, the seven inspection tools, and the bracket that changes the type.

**Task 2.1.** Build a DataFrame called `t21` with two columns, `"Mon"` and `"Tue"`, holding
`[22, 30]` and `[24, 31]`, and row labels `["Toronto", "Cairo"]`.

In [11]:
t21 = pd.DataFrame(data={"Mon" : [22,30], "Tue" : [24,31]}, index=["Toronto", "Cairo"])

In [12]:
check('2.1 shape',   lambda: t21.shape, (2, 2))
check('2.1 columns', lambda: list(t21.columns), ['Mon', 'Tue'])
check('2.1 index',   lambda: list(t21.index), ['Toronto', 'Cairo'])
check('2.1 values',  lambda: t21.to_numpy().tolist(), [[22, 24], [30, 31]])

PASS  2.1 shape
PASS  2.1 columns
PASS  2.1 index
PASS  2.1 values


**Task 2.2.** Three of the seven inspection tools, on `weather`. Shape into `t22_shape`,
the column labels as a **list** into `t22_cols`, and the number of rows into `t22_rows`.
Get `t22_rows` from the shape — do not type `4`.

In [13]:
t22_shape = weather.shape
t22_cols  = weather.columns
t22_rows  = len(weather.index)

In [14]:
check('2.2 shape',   t22_shape, (4, 7))
check('2.2 columns', t22_cols, days)
check('2.2 rows',    t22_rows, 4)

PASS  2.2 shape
PASS  2.2 columns
PASS  2.2 rows


**Task 2.3.** The bracket trap. Put the `"Wed"` column **as a Series** in `t23_series`, and
the same column **as a one-column DataFrame** in `t23_frame`.

In [15]:
t23_series = pd.Series(weather.Wed)
t23_frame  = pd.DataFrame(weather.Wed)

In [16]:
check('2.3 series type',   lambda: type(t23_series).__name__, 'Series')
check('2.3 series shape',  lambda: t23_series.shape, (4,))
check('2.3 series name',   lambda: t23_series.name, 'Wed')
check('2.3 series values', lambda: t23_series.tolist(), [25, 33, 19, 26])
check('2.3 frame type',    lambda: type(t23_frame).__name__, 'DataFrame')
check('2.3 frame shape',   lambda: t23_frame.shape, (4, 1))
check('2.3 frame column',  lambda: list(t23_frame.columns), ['Wed'])
check('2.3 frame values',  lambda: t23_frame['Wed'].tolist(), [25, 33, 19, 26])

PASS  2.3 series type
PASS  2.3 series shape
PASS  2.3 series name
PASS  2.3 series values
PASS  2.3 frame type
PASS  2.3 frame shape
PASS  2.3 frame column
PASS  2.3 frame values


**Task 2.4.** Pull the DataFrame apart into its three pieces: the raw NumPy block into
`t24_values`, the row labels as a list into `t24_index`, and the column labels as a list
into `t24_cols`.

In [17]:
t24_values = weather.values
t24_index  = weather.index
t24_cols   = weather.columns

In [18]:
check('2.4 values shape', lambda: t24_values.shape, (4, 7))
check('2.4 values type',  lambda: type(t24_values).__name__, 'ndarray')
check('2.4 values',       lambda: t24_values.tolist(),
      [[22, 24, 25, 27, 26, 25, 23], [30, 31, 33, 34, 33, 32, 30],
       [18, 19, 19, 20, 21, 20, 19], [28, 27, 26, 25, 26, 27, 28]])
check('2.4 index', t24_index, cities)
check('2.4 columns', t24_cols, days)

PASS  2.4 values shape
PASS  2.4 values type
PASS  2.4 values
PASS  2.4 index
PASS  2.4 columns


**Task 2.5.** Chapter 3's `axis` rule, on a DataFrame. Put one mean **per city** (rounded to
2 decimals) in `t25_city`, and one mean **per day** (rounded to 2) in `t25_day`. Both must
come back as labelled Series.

In [19]:
t25_city = np.round(np.mean(weather, axis=1),2)
t25_day  = np.mean(weather, axis=0)

In [20]:
check('2.5 per city index',  lambda: list(t25_city.index), cities)
check('2.5 per city values', lambda: t25_city.tolist(), [24.57, 31.86, 19.43, 26.71])
check('2.5 per day index',   lambda: list(t25_day.index), days)
check('2.5 per day values',  lambda: t25_day.tolist(), [24.5, 25.25, 25.75, 26.5, 26.5, 26.0, 25.0])

PASS  2.5 per city index
PASS  2.5 per city values
PASS  2.5 per day index
PASS  2.5 per day values


---
## Part 3 — .loc and .iloc  ·  §4.4

The most error-prone topic in the chapter. Take it slowly; Task 3.5 is the one to think
hardest about.

**Task 3.1.** Sydney's Friday high, twice: by label into `t31_loc`, by position into
`t31_iloc`.

In [21]:
t31_loc  = weather.loc["Sydney"].Fri
t31_iloc = weather.iloc[3].iloc[2]

In [22]:
check('3.1 by label',    t31_loc, 26)
check('3.1 by position', t31_iloc, 26)

PASS  3.1 by label
PASS  3.1 by position


**Task 3.2.** A whole row and a whole column. Tokyo's week into `t32_row`, and every city's
Thursday into `t32_col`. Both come back as Series — check what each one is indexed by.

In [23]:
t32_row = weather.loc["Tokyo"]
t32_col = weather.loc[::,"Thu"]

In [24]:
check('3.2 row values', lambda: t32_row.tolist(), [18, 19, 19, 20, 21, 20, 19])
check('3.2 row index',  lambda: list(t32_row.index), days)
check('3.2 col values', lambda: t32_col.tolist(), [27, 34, 20, 25])
check('3.2 col index',  lambda: list(t32_col.index), cities)

PASS  3.2 row values
PASS  3.2 row index
PASS  3.2 col values
PASS  3.2 col index


**Task 3.3.** A sub-table. Put the Toronto and Tokyo rows, Monday and Sunday columns only,
into `t33`. The result should be 2 rows by 2 columns, in that order.

In [40]:
t33 = weather.loc[["Toronto", "Tokyo"], ["Mon", "Sun"]]

In [41]:
check('3.3 shape',   lambda: t33.shape, (2, 2))
check('3.3 index',   lambda: list(t33.index), ['Toronto', 'Tokyo'])
check('3.3 columns', lambda: list(t33.columns), ['Mon', 'Sun'])
check('3.3 values',  lambda: t33.to_numpy().tolist(), [[22, 23], [18, 19]])

PASS  3.3 shape
PASS  3.3 index
PASS  3.3 columns
PASS  3.3 values


**Task 3.4.** Slice rows both ways and store the **row count** each returns. `t34_loc` from
the label slice `"Toronto"` to `"Tokyo"`. `t34_iloc` from the position slice `0` to `3`.
Then answer in `t34_same`: do they return the same number of rows? `True` or `False`.

In [54]:
t34_loc  = len(weather.loc["Toronto" : "Tokyo"])
t34_iloc = len(weather.iloc[0:3])
t34_same = True

In [55]:
check('3.4 label slice rows',    t34_loc, 3)
check('3.4 position slice rows', t34_iloc, 3)
check('3.4 same count?',         t34_same, True)

PASS  3.4 label slice rows
PASS  3.4 position slice rows
PASS  3.4 same count?


**Task 3.5.** *The one to think about.* Put `weather.loc[:, "Wed":"Mon"]` into `t35_backward`
and its shape into `t35_shape`. Then, in `t35_fixed`, produce the same three columns —
Wed, Tue, Mon, **in that order** — in a way that actually works.

In [164]:
t35_backward = weather.loc[:, "Wed":"Mon"]
t35_shape    = np.shape(t35_backward)
t35_fixed    = weather.loc[:,["Wed", "Tue", "Mon"]]

(4, 0)


In [73]:
check('3.5 backward shape', t35_shape, (4, 0))
check('3.5 fixed columns',  lambda: list(t35_fixed.columns), ['Wed', 'Tue', 'Mon'])
check('3.5 fixed shape',    lambda: t35_fixed.shape, (4, 3))
check('3.5 fixed values',   lambda: t35_fixed.to_numpy().tolist(),
      [[25, 24, 22], [33, 31, 30], [19, 19, 18], [26, 27, 28]])

PASS  3.5 backward shape
PASS  3.5 fixed columns
PASS  3.5 fixed shape
PASS  3.5 fixed values


---
## Part 4 — Filtering and sorting  ·  §4.5

Build a mask, use the mask. Then put the rows in order without losing their names.

**Task 4.1.** Put the boolean Series for "Monday above 25" in `t41_mask`, and the number of
cities that qualify in `t41_count`. Get the count from the mask — do not type it.

In [95]:

t41_mask = weather["Mon"] > 25
t41_count = t41_mask.sum()

In [96]:
check('4.1 mask values', lambda: t41_mask.tolist(), [False, True, False, True])
check('4.1 mask index',  lambda: list(t41_mask.index), cities)
check('4.1 count',       t41_count, 2)

PASS  4.1 mask values
PASS  4.1 mask index
PASS  4.1 count


**Task 4.2.** Use that idea to put the rows where Sunday is at least 28 into `t42`.

In [110]:
t42 = weather.loc[weather["Sun"] >= 28 ]
print(t42)

        Mon  Tue  Wed  Thu  Fri  Sat  Sun
Cairo    30   31   33   34   33   32   30
Sydney   28   27   26   25   26   27   28


In [117]:
check('4.2 index', lambda: list(t42.index), ['Cairo', 'Sydney'])
check('4.2 shape', lambda: t42.shape, (2, 7))
check('4.2 rows',  lambda: t42, weather.loc[['Cairo', 'Sydney']])

PASS  4.2 index
PASS  4.2 shape
PASS  4.2 rows


**Task 4.3.** Two conditions. Put the rows where Wednesday is above 20 **and** Saturday is
below 30 into `t43_and`, and the rows where Monday is below 20 **or** Friday is above 32
into `t43_or`. Remember the parentheses.

In [115]:
t43_and = weather.loc[(weather["Wed"] > 20) & (weather["Sat"] < 30)]
t43_or  = weather.loc[(weather["Mon"] < 20) | (weather["Fri"] > 32)]

In [116]:
check('4.3 and',      lambda: list(t43_and.index), ['Toronto', 'Sydney'])
check('4.3 and rows', lambda: t43_and, weather.loc[['Toronto', 'Sydney']])
check('4.3 or',       lambda: list(t43_or.index), ['Cairo', 'Tokyo'])
check('4.3 or rows',  lambda: t43_or, weather.loc[['Cairo', 'Tokyo']])

PASS  4.3 and
PASS  4.3 and rows
PASS  4.3 or
PASS  4.3 or rows


**Task 4.4.** Filter and select in one move. Put the **Thursday values** for the cities whose
Thursday is above 24 into `t44`, using the `.loc[mask, column]` form.

In [168]:
t44 = weather.loc[weather["Thu"] > 24, "Thu"]


In [169]:
check('4.4 index',  lambda: list(t44.index), ['Toronto', 'Cairo', 'Sydney'])
check('4.4 values', lambda: t44.tolist(), [27, 34, 25])
check('4.4 type',   lambda: type(t44).__name__, 'Series')

PASS  4.4 index
PASS  4.4 values
PASS  4.4 type


**Task 4.5.** Sort `weather` by Sunday, coldest first, into `t45_sorted`. Put the label of
the row that ends up **first** in `t45_first`. Then put `weather`'s own first row label in
`t45_original` — this one is a guard, and it should still be Toronto.

In [181]:
t45_sorted   = weather.sort_values(by="Sun")
t45_first    = t45_sorted.index[0]
t45_original = weather.index[0]

In [182]:
check('4.5 sorted order', lambda: list(t45_sorted.index), ['Tokyo', 'Toronto', 'Sydney', 'Cairo'])
check('4.5 first',        t45_first, 'Tokyo')
check('4.5 original first row', t45_original, 'Toronto')
check('4.5 GUARD weather intact', list(weather.index), cities)

PASS  4.5 sorted order
PASS  4.5 first
PASS  4.5 original first row
PASS  4.5 GUARD weather intact


---
## Part 5 — Reading and writing files  ·  §4.6

The setup cell wrote four CSV files beside this notebook. Now load them — including the
two that will not load correctly without help.

**Task 5.1.** Load `weather.csv` into `t51` using the `city` column as the row labels.

In [185]:
t51 = pd.read_csv("weather.csv",index_col="city")

In [186]:
check('5.1 shape',   lambda: t51.shape, (4, 7))
check('5.1 index',   lambda: list(t51.index), cities)
check('5.1 columns', lambda: list(t51.columns), days)
check('5.1 matches weather', lambda: t51, weather)

PASS  5.1 shape
PASS  5.1 index
PASS  5.1 columns
PASS  5.1 matches weather


**Task 5.2.** Two files that need an argument. Load `euro_format.csv` into `t52_euro` — it
uses semicolons. Load `with_preamble.csv` into `t52_pre` — it has two junk lines before the
header. Both should end up with sensible column names.

In [188]:
t52_euro = pd.read_csv("euro_format.csv",delimiter=";")
t52_pre  = pd.read_csv("with_preamble.csv",skiprows=2)

In [189]:
check('5.2 euro shape',   lambda: t52_euro.shape, (4, 3))
check('5.2 euro columns', lambda: list(t52_euro.columns), ['city', 'temperature', 'humidity'])
check('5.2 preamble shape',   lambda: t52_pre.shape, (4, 4))
check('5.2 preamble columns', lambda: list(t52_pre.columns), ['city', 'Mon', 'Tue', 'Wed'])

PASS  5.2 euro shape
PASS  5.2 euro columns
PASS  5.2 preamble shape
PASS  5.2 preamble columns


**Task 5.3.** Load `messy.csv` twice. Plain, into `t53_raw`. Then again into `t53_clean`,
telling Pandas that `"unknown"` and `"missing"` also mean a missing value. Put the dtype of
`t53_clean`'s `Mon` column, as a string, in `t53_dtype`.

*The checks here expect pandas 2.x, which is what gblearn runs — the setup cell printed your version. Pandas 3 names a text column `str` instead of `object`.*

In [197]:
t53_raw   = pd.read_csv("messy.csv")
t53_clean = pd.read_csv("messy.csv", na_values=["unknown", "missing"])
t53_dtype = t53_clean["Mon"].dtypes

In [198]:
check('5.3 raw Mon dtype',   lambda: str(t53_raw['Mon'].dtype), 'object')
check('5.3 clean Mon dtype', t53_dtype, 'float64')
check('5.3 clean NaN count', lambda: int(t53_clean.isna().sum().sum()), 3)

NOT YET  5.3 raw Mon dtype
        you gave: 'str'
        expected: 'object'
PASS  5.3 clean Mon dtype
PASS  5.3 clean NaN count


**Task 5.4.** Save and reload. Write `enrollments` to `"my_enrollments.csv"` **without** the
index, then read it straight back into `t54`. The round trip must return all five columns.

In [201]:
# write it, then read it back
enrollments.to_csv("my_enrollments.csv", index=False)
t54 = pd.read_csv("my_enrollments.csv")

In [202]:
check('5.4 shape',   lambda: t54.shape, (6, 5))
check('5.4 columns', lambda: list(t54.columns),
      ['course', 'section', 'enrolled', 'capacity', 'delivery'])

PASS  5.4 shape
PASS  5.4 columns


---
## Part 6 — Putting it together  ·  §4.7

Two questions of the kind a registrar actually asks. Load, filter, answer.

**Task 6.1.** Which sections are over capacity? Compare the `enrolled` column directly
against the `capacity` column and put the matching rows in `t61`. Put the course code of the
single match in `t61_course`.

In [227]:
enrollments = pd.read_csv("my_enrollments.csv")
print(enrollments)
t61        = enrollments.loc[enrollments["enrolled"] > enrollments["capacity"]]
print(t61)
t61_course = t61.iloc[0].course
print(t61_course)

     course section  enrolled  capacity   delivery
0  COMP3122       A        38        40  in-person
1  COMP3122       B        42        40  in-person
2  COMP1011       A        35        40     online
3  COMP1011       B        28        40     online
4  COMP2152       A        40        40     hybrid
5  COMP2152       B        22        40     hybrid
     course section  enrolled  capacity   delivery
1  COMP3122       B        42        40  in-person
COMP3122


In [228]:
check('6.1 rows',   lambda: t61.shape[0], 1)
check('6.1 course', t61_course, 'COMP3122')
check('6.1 section', lambda: t61['section'].iloc[0], 'B')

PASS  6.1 rows
PASS  6.1 course
PASS  6.1 section


**Task 6.2.** Put the average enrolment across all six sections, rounded to 2 decimals, in
`t62_mean`. Then put the `course` values of the online sections, as a list, in `t62_online`.

In [244]:
t62_mean   = np.round(np.mean(enrollments.enrolled),2)
t62_online = enrollments.loc[enrollments["delivery"] == "online", "course"].to_list

In [245]:
check('6.2 mean',   t62_mean, 34.17)
check('6.2 online', t62_online, ['COMP1011', 'COMP1011'])

PASS  6.2 mean
PASS  6.2 online


---
## Part 7 — Explore a real dataset  ·  not graded

Every table above was built in the notebook so that each check has one clean answer. This part uses **real observations**: 344 penguins measured at Palmer Station, Antarctica, in 2007–2009. Source: Gorman K.B., Williams T.D., Fraser W.R. (2014), *Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus Pygoscelis)*, PLoS ONE 9(3): e90081; data collected by the Palmer Station Long Term Ecological Research program and released under CC0 as the `palmerpenguins` dataset. The file `penguins.csv` is in this week's website bundle (`fetch_week(4)` puts it in `practice/week04/`); the cell below reads it from the site, or from beside this notebook.

Seven columns: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`. There are no checks here — your lab professor reads your answers.

**Task 7.1.** Load the file into `penguins` and run the seven inspection tools on it. Then
answer, each with one Pandas expression (no loops):

- `t71_rows` — how many penguins were measured?
- `t71_missing` — which columns have missing values? Read it off `info()` and type the
  column names as a list.
- `t71_gentoo` — the mean body mass, in grams, of the Gentoo penguins, rounded to 1 decimal.
  (Filter with `.loc[mask, column]`; one species at a time — `groupby` is Week 12.)
- `t71_heaviest` — the `species` of the heaviest penguin in the table.

Finish with two sentences as comments at the end of your answer cell: one **decision** the
numbers support (a field team with one set of scales — which species do they weigh first, and
why?) and one **limitation** of the data (what the missing values or the years mean for that
decision).

In [255]:
PENGUINS = "https://comp3122.gblearn.com/notebooks/week04/penguins.csv"  # or "penguins.csv" if the file is beside this notebook
penguins = pd.read_csv(PENGUINS)

t71_rows     = len(penguins)
t71_missing  = penguins.info()
t71_gentoo   = np.round(penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"].mean(), 1)
t71_heaviest = penguins.loc[penguins["body_mass_g"] == np.max(penguins.body_mass_g), "species"]
print(t71_heaviest)

# Decision:
# Limitation:

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB
237    Gentoo
Name: species, dtype: str


In [257]:
print('Part 7 has no check. Your lab professor reads your answer cell and your two sentences.')

Part 7 has no check. Your lab professor reads your answer cell and your two sentences.


---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done.

Then read back through your **answer** cells — the ones you typed in — and look for `for`. If one is there, that task is not finished, however right the numbers look. The checks cannot see a loop; the marker can.

Then hand it in as *How to submit a lab — read once* at the top of 02 · Lab on D2L says: run the notebook's last cell, *Your submission line*, and paste the line it prints into *Lab 4 — submit*. Nothing else.

**If you got stuck on Task 3.5**, that is the intended difficulty. A backward label slice returns an empty result and raises nothing at all — which is worse than an error, because your code keeps running. Re-read §4.4 and bring the question to class.

**One habit to take with you.** Every time you meet a new DataFrame for the rest of this course, run the seven: `shape`, `head`, `tail`, `info`, `describe`, `dtypes`, `columns`. It takes four seconds and it catches the mistakes that otherwise cost an afternoon.

---
## Your submission line

Run the cell below and paste **the line it prints** into *Lab 4 — submit* on D2L. Nothing else — not the address bar of this notebook, which carries a port number that stops working when your server restarts. The line is built from your account name, so it always works.

In [258]:
# Run this cell. Copy the line it prints into 'Lab 4 -- submit' on D2L. Nothing else.
from pathlib import Path
account = Path.home().name                                   # your gblearn account name
print(f'http://{account}.gblearn.com/python/?open=comp3122/labs/lab04/COMP3122-Lab04-Student.ipynb')

http://chris.gblearn.com/python/?open=comp3122/labs/lab04/COMP3122-Lab04-Student.ipynb
